# 03_Visualization

## 日本語
このNotebookは、論文再現のための3段階ワークフローの第3部であり、論文で使用した地図図表を作成するためのものです。

../data/raw/municipality_shapefile/ から市区町村シェープファイルを読み込み、研究対象地域および地域区分に関する論文用の地図を作成します。

このNotebookでは、少なくとも本四架橋ネットワークと研究対象地域の地図、および地域間産業連関分析で用いる全国地域区分地図を出力します。

生成された図は ../output/ 以下に保存されます。

## English
This notebook is the third part of the three-step reproduction workflow for the paper. This notebook generates the map figures used in the paper.

It reads the municipality shapefile from ../data/raw/municipality_shapefile/ and creates publication-ready maps for the study area and regional classification.

The notebook includes at least two outputs: a map of the Honshu–Shikoku bridge network and study area, and a map of Japan’s regional classification used in the interregional input–output analysis.

Generated figures are saved under ../output/.


In [1]:
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patches as mpatches
from matplotlib.offsetbox import TextArea, VPacker, AnnotationBbox
import os

## GIS作成（初回1回）

In [ ]:
# --- パス設定 ---
SHAPEFILE_PATH = "../data/raw/municipality_shapefile/region.shp"
SAVE_DIR = '../output'
PROCESSED_SHP_DIR = os.path.join(SAVE_DIR, "processed_map")
PROCESSED_SHP_PATH = os.path.join(PROCESSED_SHP_DIR, "regions_dissolved.shp")

os.makedirs(PROCESSED_SHP_DIR, exist_ok=True)

def get_preprocessed_gdf():
# 処理済みのファイルがあれば読み込み、なければ作成して保存する関数
    if os.path.exists(PROCESSED_SHP_PATH):
        print(f"Loading preprocessed map from: {PROCESSED_SHP_PATH}")
        gdf = gpd.read_file(PROCESSED_SHP_PATH)
    else:
        print("Pre-processing GeoData for the first time...")
        # 元データを読み込み、47都道府県に限定してdissolve
        gdf_raw = gpd.read_file(SHAPEFILE_PATH)[:47]
        gdf = gdf_raw.dissolve(by='region').reset_index()

        # Googleドライブに保存
        print(f"Saving preprocessed map to: {PROCESSED_SHP_PATH}")
        gdf.to_file(PROCESSED_SHP_PATH, driver='ESRI Shapefile', encoding='utf-8')

    return gdf

gdf_regions_preprocessed = get_preprocessed_gdf() # 実行

In [ ]:
# 1. ファイルの存在確認
if os.path.exists(PROCESSED_SHP_PATH):
    stats = os.stat(PROCESSED_SHP_PATH)
    print(f"✅ ファイルを確認しました: {PROCESSED_SHP_PATH}")
    print(f"   サイズ: {stats.st_size / 1024:.2f} KB")

    # 2. 再読み込みテスト
    test_gdf = gpd.read_file(PROCESSED_SHP_PATH)
    print(f"   地域数: {len(test_gdf)} (期待値: 9地域)")
    print(f"   含まれる地域: {test_gdf['region'].tolist()}")

    # 3. 簡易プロットで形状確認
    test_gdf.plot(edgecolor='black', facecolor='lightblue')
    plt.title("Preprocessed Regions (Dissolved)")
    plt.show()
else:
    print("❌ ファイルが見つかりません。パスを確認してください。")

## 本四架橋図

In [ ]:
# ==========================================
# --- 設定切り替えスイッチ ---
# True: カラーモード / False: 白黒モード
COLOR_MODE = True
# ==========================================

# --- 0. フォント設定 ---
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

# --- 色設定の定義 ---
if COLOR_MODE:
    c = {
        'bg_map': '#F4FAFF', 'land_shikoku': '#FFFACD', 'land_other': '#E6EEF5',
        'boundary_edge': '#99AABB', 'shikoku_edge': '#556677',
        'txt_pref': '#003366', 'txt_city': 'black', 'txt_shikoku': '#003366',
        'grid': '#B0C4DE', 'inset_land': '#E6E6E6', 'inset_rect': '#E60012',
        'bridge_base': 'white',
        'route_colors': {'Onomichi': '#00A651', 'Kojima': '#E60012', 'Kobe': '#0068B7'},
        'bbox_ec': '#555555'
    }
else:
    c = {
        'bg_map': 'white', 'land_shikoku': '#ffffff', 'land_other': '#f2f2f2',
        'boundary_edge': '#999999', 'shikoku_edge': '#555555',
        'txt_pref': '#333333', 'txt_city': 'black', 'txt_shikoku': '#444444',
        'grid': '#cccccc', 'inset_land': '#dddddd', 'inset_rect': 'black',
        'bridge_base': 'white',
        'route_colors': {'Onomichi': 'black', 'Kojima': 'black', 'Kobe': 'black'},
        'bbox_ec': 'black'
    }

# --- 1. データ準備 ---
shapefile_path = "../data/raw/municipality_shapefile/region.shp"
try:
    gdf = gpd.read_file(shapefile_path).iloc[:47]
except FileNotFoundError:
    from shapely.geometry import Polygon
    gdf = gpd.GeoDataFrame({'region': ['Other', '四国'], 'geometry': [Polygon([(130,30), (140,30), (140,40), (130,40)]), Polygon([(132,32), (136,32), (135,35), (133,35)])]}, crs="EPSG:4326")

def plot_bridge(ax, coords, label, label_pos, text_offset=(0, 0), route_color='black'):
    lons, lats = zip(*coords)
    ax.plot(lons, lats, color=c['bridge_base'], linewidth=5, solid_capstyle='round', zorder=3)
    ax.plot(lons, lats, color=route_color, linewidth=2.5, solid_capstyle='round', zorder=4)

    lines = label.split('\n')
    bold_text = '\n'.join(lines[:2])
    regular_text = '\n'.join(lines[2:])

    # --- フォントサイズを 10 にアップ ---
    t1 = TextArea(bold_text, textprops=dict(fontweight='bold', fontsize=12, family='serif', color=c['txt_city']))
    t2 = TextArea(regular_text, textprops=dict(fontweight='normal', fontsize=12, family='serif', color=c['txt_city']))

    packer = VPacker(children=[t1, t2], align="center", pad=0, sep=2)
    box_ec = route_color if COLOR_MODE else c['bbox_ec']
    arrow_color = route_color if COLOR_MODE else 'black'

    ab = AnnotationBbox(packer, label_pos, xybox=text_offset,
                        xycoords='data', boxcoords='offset points',
                        arrowprops=dict(arrowstyle='-', connectionstyle='arc3,rad=0.2', color=arrow_color, lw=1.0),
                        bboxprops=dict(boxstyle='round,pad=0.4', fc='white', ec=box_ec, lw=1.2 if COLOR_MODE else 0.8, alpha=0.9),
                        zorder=7)
    ax.add_artist(ab)

# --- 2. 定義 ---
bridge_routes = {
    'Onomichi–Imabari Route\n(Shimanami Kaido)\nExpressway only\nCompleted 1999': {
        'coords': [[133.239166,34.461957],[133.235733,34.448934],[133.21788,34.431946],[133.21994,34.419485],[133.193161,34.382659],[133.193161,34.362256],[133.159515,34.335045],[133.155396,34.310093],[133.144409,34.299316],[133.094284,34.265842],[133.047592,34.250519],[133.047592,34.224975],[133.079865,34.205669],[133.064758,34.167613],[133.053772,34.136929],[133.031799,34.130677],[132.978241,34.112487],[132.977554,34.096567],[132.971375,34.053342]],
        'label_pos': (133.05, 34.2), 'text_offset': (-105, -5), 'color_key': 'Onomichi'
    },
    'Kojima–Sakaide Route\n(Great Seto Bridge)\nRoad & Railway\nCompleted 1988': {
        'coords': [[133.793427,34.626597],[133.811966,34.602015],[133.79995,34.543498],[133.80304,34.530771],[133.780724,34.476449],[133.801323,34.451539],[133.808533,34.440214],[133.804756,34.421241],[133.816326,34.389006],[133.828857,34.354859],[133.826283,34.338135],[133.828514,34.325235],[133.835896,34.319706],[133.844822,34.297444],[133.851517,34.282127]],
        'label_pos': (133.81, 34.4), 'text_offset': (-100, 60), 'color_key': 'Kojima'
    },
    'Kobe–Naruto Route\n(Akashi-Kaikyo/Naruto)\nExpressway only\nCompleted 1998': {
        'coords': [[135.085762,34.728719],[135.092663,34.71595],[135.073265,34.691324],[135.067514,34.658995],[135.0355,34.63414],[135.007519,34.600588],[135.016188,34.582005],[134.981907,34.544487],[134.892471,34.521295],[134.872559,34.512809],[134.873417,34.442762],[134.859856,34.373224],[134.84063,34.342189],[134.816597,34.32688],[134.751537,34.311993],[134.683319,34.259543],[134.637726,34.234257],[134.627254,34.221625],[134.611118,34.18954],[134.574726,34.174061],[134.563911,34.170937]],
        'label_pos': (134.8, 34.45), 'text_offset': (-40, 70), 'color_key': 'Kobe'
    }
}

prefecture_labels = {'Ehime': (132.9, 33.7), 'Kagawa': (133.9, 34.20), 'Tokushima': (134.3, 33.9), 'Kochi': (133.1, 33.4)}
cities = {'Matsuyama': (132.76, 33.84), 'Takamatsu': (134.04, 34.34), 'Tokushima': (134.56, 34.07), 'Kochi': (133.53, 33.56), 'Hiroshima': (132.45, 34.38), 'Okayama': (133.92, 34.66), 'Osaka': (135.50, 34.69), 'Kobe': (135.19, 34.69)}

# --- 3. メインプロット ---
fig, ax = plt.subplots(figsize=(12, 10))
ax.set_facecolor(c['bg_map'])
fig.patch.set_facecolor('white')

gdf['color'] = np.where(gdf['region'] == '四国', c['land_shikoku'], c['land_other'])
gdf.plot(ax=ax, color=gdf['color'], edgecolor=c['boundary_edge'], linewidth=0.5, zorder=1)
gdf[gdf['region'] == '四国'].geometry.boundary.plot(ax=ax, color=c['shikoku_edge'], linewidth=1.0, zorder=2)

# --- 県名ラベルを大きく (11 -> 14) ---
for pref, (lon, lat) in prefecture_labels.items():
    ax.text(lon, lat, pref, fontsize=14, fontweight='bold', color=c['txt_pref'],
            ha='center', va='center', alpha=0.4 if COLOR_MODE else 0.3, zorder=2)

for name, data in bridge_routes.items():
    route_color = c['route_colors'][data['color_key']]
    plot_bridge(ax, data['coords'], name, data['label_pos'], data['text_offset'], route_color=route_color)

# --- 都市名ラベルを大きく (9 -> 11) ---
for city, (lon, lat) in cities.items():
    ax.plot(lon, lat, 'o', color=c['txt_city'], markersize=6, markeredgecolor='white', zorder=6)
    kwargs = dict(fontsize=11, zorder=8, color=c['txt_city'])
    if city == 'Matsuyama': ax.text(lon + 0.05, lat, city, ha='left', va='center', fontweight='bold', **kwargs)
    elif city == 'Takamatsu': ax.text(lon, lat - 0.05, city, ha='center', va='top', fontweight='bold', **kwargs)
    elif city == 'Tokushima': ax.text(lon - 0.05, lat, city, ha='right', va='center', fontweight='bold', **kwargs)
    elif city == 'Osaka': ax.text(lon + 0.05, lat, city, ha='left', va='center', fontweight='bold', **kwargs)
    elif city in ['Kobe', 'Kochi']: ax.text(lon, lat + 0.05, city, ha='center', va='bottom', fontweight='bold' if city=='Kochi' else 'normal', **kwargs)
    else: ax.text(lon, lat + 0.05, city, ha='center', **kwargs)

# --- SHIKOKU ロゴを大きく (18 -> 24) ---
ax.text(133.6, 33.75, 'SHIKOKU', fontsize=24, fontweight='bold', fontstyle='italic', color=c['txt_shikoku'], alpha=0.5, ha='center', va='center', zorder=1)

# --- 4. レイアウトの調整 ---
ax.set_xlim(131.5, 136.0)
ax.set_ylim(32.4, 35.0)
ax.set_xticks(np.arange(132, 137, 1))
ax.set_yticks(np.arange(33, 36, 1))
ax.set_xticklabels([f'{x}$^\circ$E' for x in np.arange(132, 137, 1)]) # Escaping the backslash
ax.set_yticklabels([f'{y}$^\circ$N' for y in np.arange(33, 36, 1)]) # Escaping the backslash
# 軸の数字サイズを調整
ax.tick_params(axis='both', which='major', labelsize=11)
ax.grid(True, linestyle='--', color=c['grid'], alpha=0.5, zorder=0)

# ==========================================
# --- 日本地図インセット (修正ポイント) ---
# ==========================================
ax_ins = inset_axes(ax, width="25%", height="25%", loc='lower right', borderpad=3)
ax_ins.set_facecolor('white') # インセット背景を少し明るくして独立させる

# 陸地を描画する際に edgecolor（境界線）を追加
gdf.plot(ax=ax_ins, color=c['inset_land'], edgecolor='#777777', linewidth=0.3)

ax_ins.set_xlim(122, 149); ax_ins.set_ylim(24, 46)
for name, (lon, lat) in {'Tokyo': (139.69, 35.68), 'Osaka': (135.50, 34.69)}.items():
    ax_ins.plot(lon, lat, 'o', color='black', markersize=2)
    ax_ins.text(lon, lat + 1.5, name, fontsize=8, ha='center')


rect = mpatches.Rectangle((131.5, 32.4), 4.5, 2.6, linewidth=1.2, edgecolor=c['inset_rect'], facecolor='none', zorder=10)
ax_ins.add_patch(rect)
ax_ins.set_xticks([]); ax_ins.set_yticks([])

# --- スケールバーと方位記号 ---
bar_x, bar_y = 131.8, 32.55
ax.plot([bar_x, bar_x + 1.08], [bar_y, bar_y], color='black', linewidth=2, zorder=10)
ax.text(bar_x + 0.54, bar_y + 0.05, '100 km', ha='center', va='bottom', fontsize=10, color='black')
ax.annotate('', xy=(131.8, 34.85), xytext=(131.8, 34.5), arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
ax.text(131.8, 34.9, 'N', ha='center', va='bottom', fontsize=12, fontweight='bold', color='black')

plt.tight_layout()
# ファイル保存名の決定
filename = 'shikoku_bridges_color.png' if COLOR_MODE else 'shikoku_bridges_bw.png'
plt.savefig(filename, bbox_inches='tight', dpi=300)
plt.show()

## 全国地域区分地図

### カラー版

In [ ]:
from matplotlib.patches import Rectangle, FancyArrowPatch
import os
import geopandas as gpd
import matplotlib.pyplot as plt

# --- パス設定 ---
SHAPEFILE_PATH = "../data/raw/municipality_shapefile/region.shp"
SAVE_DIR = '../output'
PROCESSED_SHP_DIR = os.path.join(SAVE_DIR, "processed_map")
PROCESSED_SHP_PATH = os.path.join(PROCESSED_SHP_DIR, "regions_dissolved.shp")

# ディレクトリが存在しない場合は作成
os.makedirs(SAVE_DIR, exist_ok=True)

# データ読み込み
try:
    gdf = gpd.read_file(PROCESSED_SHP_PATH)
except FileNotFoundError:
    print(f"エラー: ファイルが見つかりません。パスを確認してください: {PROCESSED_SHP_PATH}")
    raise SystemExit

# ----------------------------
# 配色設定（論文向け：面は薄め、境界は濃いめ）
# ----------------------------
other_region_color = '#DEDEDE'   # 薄いグレー
border_color = '#666666'         # やや濃い境界線
highlight_color = '#8E44AD'      # 四国の強調色（落ち着いた紫）

region_colors = {
    '北海道': other_region_color,
    '東北': other_region_color,
    '関東': other_region_color,
    '中部': other_region_color,
    '近畿': other_region_color,
    '中国': other_region_color,
    '四国': highlight_color,
    '九州': other_region_color,
    '沖縄': other_region_color
}

# 地域名の英語対応辞書
region_names_en = {
    '北海道': 'Hokkaido',
    '東北': 'Tohoku',
    '関東': 'Kanto',
    '中部': 'Chubu',
    '近畿': 'Kinki',
    '中国': 'Chugoku',
    '四国': 'Shikoku',
    '九州': 'Kyushu',
    '沖縄': 'Okinawa'
}

# 沖縄と本土を分離
gdf_okinawa = gdf[gdf['region'] == '沖縄'].copy()
gdf_mainland = gdf[gdf['region'] != '沖縄'].copy()

# 図の作成
fig = plt.figure(figsize=(14, 10), facecolor='white')
ax_main = fig.add_axes([0.1, 0.1, 0.8, 0.8])

# ----------------------------
# メインランド描画
# 地域ごとに1回ずつ描く
# ----------------------------
for region in gdf_mainland['region'].unique():
    region_data = gdf_mainland[gdf_mainland['region'] == region]
    region_en = region_names_en.get(region, region)
    color = region_colors.get(region, '#DEDEDE')

    region_data.plot(
        ax=ax_main,
        color=color,
        edgecolor=border_color,
        linewidth=1.4,
        alpha=1.0
    )

    centroid = region_data.geometry.union_all().centroid
    ax_main.text(
        centroid.x, centroid.y, region_en,
        fontsize=16, fontweight='bold',
        ha='center', va='center',
        bbox=dict(
            boxstyle='round,pad=0.35',
            facecolor='white',
            alpha=0.9,
            edgecolor='none'
        )
    )

ax_main.set_ylim(bottom=30.5, top=45.7)
ax_main.set_xlim(left=128.0, right=146.0)
ax_main.axis('off')

# ----------------------------
# 沖縄インセット
# ----------------------------
if not gdf_okinawa.empty:
    inset_x = 0.20
    inset_y = 0.55
    inset_w = 0.20
    inset_h = 0.20

    ax_inset = fig.add_axes([inset_x, inset_y, inset_w, inset_h])

    gdf_okinawa.plot(
        ax=ax_inset,
        color=region_colors.get('沖縄', other_region_color),
        edgecolor=border_color,
        linewidth=1.2,
        alpha=1.0
    )

    centroid_okinawa = gdf_okinawa.geometry.union_all().centroid
    ax_inset.text(
        centroid_okinawa.x, centroid_okinawa.y, 'Okinawa',
        fontsize=16, fontweight='bold',
        ha='center', va='center',
        bbox=dict(
            boxstyle='round,pad=0.35',
            facecolor='white',
            alpha=0.9,
            edgecolor='none'
        )
    )

    ax_inset.axis('off')

    # インセット枠
    rect = Rectangle(
        (inset_x, inset_y), inset_w, inset_h,
        transform=fig.transFigure,
        fill=False, edgecolor='black', linewidth=2
    )
    fig.patches.append(rect)

    # 本来の位置を示す破線矢印
    arrow_start = (0.25, 0.07)
    arrow_end = (inset_x, inset_y)

    arrow = FancyArrowPatch(
        arrow_start, arrow_end,
        transform=fig.transFigure,
        arrowstyle='<-,head_width=0.8,head_length=1.0',
        connectionstyle='arc3,rad=-0.2',
        color='black',
        linewidth=1.8,
        linestyle='--',
        mutation_scale=10,
        zorder=10
    )
    fig.patches.append(arrow)

# 保存と表示
save_path = os.path.join(SAVE_DIR, 'region_map_paper_style.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"地図を保存しました: {save_path}")
plt.show()

### グレー版

In [ ]:
# --- 色とハッチングの設定（学術論文用グレースケール） ---
# 四国：濃いグレー + 斜線
# その他：非常に薄いグレー
other_region_color = '#E6E6E6'  # 薄いグレー
shikoku_color = '#4D4D4D'       # 濃いグレー

region_colors = {k: other_region_color for k in region_names_en.keys()}
region_colors['四国'] = shikoku_color

# 図の作成
fig = plt.figure(figsize=(14, 10))
ax_main = fig.add_axes([0.1, 0.1, 0.8, 0.8])

# 本土の描画
for idx, row in gdf_mainland.iterrows():
    region = row['region']
    region_en = region_names_en.get(region, region)
    color = region_colors.get(region, '#FFFFFF')

    # 四国のみ斜線を入れる
    hatch_pattern = '////' if region == '四国' else None

    gdf_mainland[gdf_mainland['region'] == region].plot(
        ax=ax_main,
        color=color,
        edgecolor='black',  # 境界線を黒に固定
        linewidth=1.2,
        hatch=hatch_pattern,
        alpha=1.0
    )

    # ラベル背景のボックスも白黒で調整
    centroid = row.geometry.centroid
    ax_main.text(centroid.x, centroid.y, region_en,
                 fontsize=18, fontweight='bold',
                 ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.4',
                           facecolor='white', alpha=0.9, edgecolor='black'))

# インセット（沖縄）の描画
if not gdf_okinawa.empty:
    ax_inset = fig.add_axes([0.20, 0.55, 0.20, 0.20])
    gdf_okinawa.plot(ax=ax_inset, color=other_region_color, edgecolor='black', linewidth=1.2)

    centroid_okinawa = gdf_okinawa.geometry.centroid.iloc[0]
    ax_inset.text(centroid_okinawa.x, centroid_okinawa.y, 'Okinawa',
                  fontsize=18, fontweight='bold', ha='center', va='center',
                  bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9, edgecolor='black'))
    ax_inset.axis('off')

    # インセット枠と矢印（点線）
    rect = Rectangle((0.20, 0.55), 0.20, 0.20, transform=fig.transFigure, fill=False, edgecolor='black', linewidth=2)
    fig.patches.append(rect)

    arrow = FancyArrowPatch((0.25, 0.07), (0.20, 0.55), transform=fig.transFigure,
                            arrowstyle='<-,head_width=0.8,head_length=1.0',
                            connectionstyle='arc3,rad=-0.2', color='black', linewidth=2, linestyle='--')
    fig.patches.append(arrow)

ax_main.set_ylim(30.5, 45.6)
ax_main.set_xlim(128.0, 146.0)
ax_main.set_title('Spatial Architecture of METI IRIOT (9 Regions)', fontsize=20, fontweight='bold', pad=20)
ax_main.axis('off')

# 保存（PDFとPNGの両方で保存するのがおすすめ）
plt.savefig('region_map_bw.png', dpi=300, bbox_inches='tight')
plt.show()

## 構造的先鋭化の図

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# 過去の構造（幅広い輸出基盤、合計幅1.0）
widths_past = np.array([0.30, 0.20, 0.25, 0.25])
ssr_past = np.array([1.2, 0.8, 1.3, 1.1])
x_past = np.append([0], np.cumsum(widths_past)[:-1])

for i in range(len(widths_past)):
    color = 'steelblue' if ssr_past[i] > 1.0 else 'lightgray'
    axes[0].bar(x_past[i], ssr_past[i], width=widths_past[i], align='edge',
                color=color, edgecolor='black', linewidth=1.5)

axes[0].axhline(y=1.0, color='red', linestyle='--', linewidth=2)
axes[0].set_title("Past: Broad Export Base", fontsize=14)
axes[0].set_xlabel("Output Share (Width) - Total = 1.0", fontsize=12)
axes[0].set_ylabel("Self-Sufficiency Rate [SSR] (Height)", fontsize=12)
axes[0].set_xlim(0, 1.0)
axes[0].set_ylim(0, 1.8)

# 現在の構造：構造的先鋭化（輸出基盤の幅縮小とSSR上昇、合計幅1.0）
widths_now = np.array([0.15, 0.45, 0.30, 0.10])
ssr_now = np.array([1.6, 0.6, 0.4, 1.5])
x_now = np.append([0], np.cumsum(widths_now)[:-1])

for i in range(len(widths_now)):
    color = 'steelblue' if ssr_now[i] > 1.0 else 'lightgray'
    axes[1].bar(x_now[i], ssr_now[i], width=widths_now[i], align='edge',
                color=color, edgecolor='black', linewidth=1.5)

axes[1].axhline(y=1.0, color='red', linestyle='--', linewidth=2)
axes[1].set_title("Present: Structural Sharpening", fontsize=14)
axes[1].set_xlabel("Output Share (Width) - Total = 1.0", fontsize=12)
axes[1].set_xlim(0, 1.0)

# 変化を示す注釈と矢印
axes[1].annotate('Export Base Contracted\n(Blue Width ◄►)',
                 xy=(0.075, 1.05), xytext=(0.3, 1.2),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=6),
                 fontsize=11)
axes[1].annotate('SSR Sharpened\n(Blue Height ▲)',
                 xy=(0.075, 1.6), xytext=(0.2, 1.65),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=6),
                 fontsize=11)

plt.tight_layout()
plt.show()